# Search by Meaning, by Hand

I build a simple semantic search system using embeddings and cosine similarity.

Instead of using a vector database, I compute cosine similarity manually with NumPy.

The goal is to retrieve passages by meaning rather than exact keyword matching.

In [1]:
import json
import numpy as np

from sentence_transformers import SentenceTransformer

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("knowledge_base.json", "r", encoding="utf-8") as f:
    knowledge_base = json.load(f)

print("Documents:", len(knowledge_base))

for item in knowledge_base:
    print(item["id"], "-", item["source"])

Documents: 10
kb-01 - handbook.md
kb-02 - handbook.md
kb-03 - handbook.md
kb-04 - policy.md
kb-05 - policy.md
kb-06 - policy.md
kb-07 - it.md
kb-08 - it.md
kb-09 - it.md
kb-10 - facilities.md


## Load embedding model

I use the local Sentence Transformers model
`all-MiniLM-L6-v2`.

The same embedding model is used for both
documents and queries.

In [3]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6014.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
embedded_docs = []

for item in knowledge_base:

    embedding = model.encode(
        item["text"]
    )

    embedded_docs.append({
        "id": item["id"],
        "source": item["source"],
        "text": item["text"],
        "embedding": embedding
    })

print("Embedded documents:", len(embedded_docs))

Embedded documents: 10


## Cosine Similarity

Cosine similarity measures how close two vectors are in direction.

I compute it manually using NumPy instead of using a vector database search function.

In [6]:
def cosine_similarity(a, b):

    numerator = np.dot(a, b)

    denominator = (
        np.linalg.norm(a)
        *
        np.linalg.norm(b)
    )

    return numerator / denominator

In [7]:
def search(query, top_k=3):

    query_embedding = model.encode(query)

    scores = []

    for doc in embedded_docs:

        score = cosine_similarity(
            query_embedding,
            doc["embedding"]
        )

        scores.append({
            "id": doc["id"],
            "source": doc["source"],
            "text": doc["text"],
            "score": score
        })

    scores.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return scores[:top_k]

In [8]:
queries = [
    "my laptop won't switch on",
    "how do I stop being billed every month?",
    "access denied error when saving a file",
    "where do I leave my car in the evening?"
]

In [9]:
for query in queries:

    print("=" * 80)
    print("QUERY:")
    print(query)
    print()

    results = search(query)

    for rank, result in enumerate(results, start=1):

        print(
            f"{rank}. "
            f"{result['id']} "
            f"({result['source']})"
        )

        print(
            f"Score: "
            f"{result['score']:.4f}"
        )

        print(result["text"])
        print()

QUERY:
my laptop won't switch on

1. kb-02 (handbook.md)
Score: 0.4369
To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.

2. kb-07 (it.md)
Score: 0.1774
Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.

3. kb-09 (it.md)
Score: 0.1581
Company laptops back up automatically to the cloud every night at 2am while connected to the office network. Files in the Desktop and Documents folders are included; external drives are not.

QUERY:
how do I stop being billed every month?

1. kb-05 (policy.md)
Score: 0.5408
To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.

2. kb-06 (policy.md)
Score: 0.3041
Premium plan members get priority support, with a

## Reflection

For all four queries, the correct passage appeared as the top result.

In several cases, the query and the retrieved passage did not share the same words. For example, the query "my laptop won't switch on" matched a passage about a device that "won't turn on". Likewise, the query about stopping monthly billing matched a passage about cancelling a subscription.

This shows that embeddings capture semantic meaning rather than only exact keyword overlap. The embedding model was able to recognize related concepts even when different words were used.

In [10]:
query = "what is the wifi password?"

results = search(query)

for rank, result in enumerate(results, start=1):

    print(
        f"{rank}. "
        f"{result['id']} "
        f"Score: "
        f"{result['score']:.4f}"
    )

    print(result["text"])
    print()

1. kb-07 Score: 0.3115
Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.

2. kb-09 Score: 0.1336
Company laptops back up automatically to the cloud every night at 2am while connected to the office network. Files in the Desktop and Documents folders are included; external drives are not.

3. kb-02 Score: 0.1084
To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.



## Optional Stretch

Query used:

```text
what is the wifi password?

The knowledge base does not contain any information about a Wi-Fi password. However, the search system still returned the password reset article as the best match because the embedding model considered the concepts somewhat related.

The top similarity score (0.3115) was noticeably lower than the scores of the valid queries, which were typically above 0.43–0.54.

In a real RAG system, a similarity threshold could be used to decide that no reliable answer exists. For example, if the best score is below a chosen threshold, the system could respond with "I could not find relevant information in the knowledge base" instead of returning a weak match.


```
